In [1]:
import os
from pathlib import Path

import duckdb
from dotenv import load_dotenv
import pandas as pd

In [2]:
# def infer_table_order(schema_sql):
#     """Infer table order based on foreign key dependencies."""
#     tables = {}
#     dependencies = {}
#     current_table = None
    
#     for line in schema_sql.split('\n'):
#         line = line.strip()
#         if line.startswith('CREATE TABLE'):
#             current_table = line.split()[2].strip('"')
#             tables[current_table] = set()
#             dependencies[current_table] = set()
#         elif 'FOREIGN KEY' in line and 'REFERENCES' in line:
#             if current_table is None:
#                 continue  # Skip if we're not inside a table definition
#             referenced_table = line.split('REFERENCES')[1].strip().split()[0].strip('"')
#             tables[current_table].add(referenced_table)
#             dependencies[current_table].add(referenced_table)
    
#     # Topological sort
#     def topological_sort(graph):
#         visited = set()
#         result = []
        
#         def dfs(node):
#             if node in visited:
#                 return
#             visited.add(node)
#             for neighbor in graph.get(node, []):
#                 if neighbor not in visited:
#                     dfs(neighbor)
#             result.append(node)
        
#         for node in graph:
#             if node not in visited:
#                 dfs(node)
        
#         return result[::-1]  # Reverse to get correct order
    
#     return topological_sort(dependencies)

# # The rest of the export_import_duckdb function remains the same

In [3]:
table_order = ['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']

In [4]:

# def export_import_duckdb_MD_BUG(db_name, new_database, export_dir=None, location='local', schema_file='schema.sql', table_order=None):
#     load_dotenv()

#     if location == 'motherduck':
#         md_token = os.getenv('MD_TOKEN')
#         if not md_token:
#             raise ValueError("MotherDuck token not found in .env file")
#         db_name = f"md:{db_name}?motherduck_token={md_token}"
#         new_database = f"md:{new_database}?motherduck_token={md_token}"

#     # Connect to the original database
#     conn = duckdb.connect(db_name)

#     # Get list of tables
#     tables = conn.execute("SHOW TABLES").fetchall()
#     table_names = [table[0] for table in tables]

#     # Create export directory if it doesn't exist
#     if export_dir is None:
#         export_dir = Path('duckdb_export')
#     else:
#         export_dir = Path(export_dir)
#     export_dir.mkdir(parents=True, exist_ok=True)

#     # Export tables to Parquet
#     for table in table_names:
#         parquet_file = export_dir / f"{table}.parquet"
#         conn.execute(f"COPY (SELECT * FROM {table}) TO '{parquet_file}' (FORMAT 'parquet')")

#     conn.close()

#     if True:
#         Path(new_database).unlink(missing_ok=True)

#     # Connect to the new database
#     new_conn = duckdb.connect(new_database)

#     # Execute the schema SQL to create tables
#     with open(schema_file, 'r') as f:
#         schema_sql = f.read()
#         new_conn.execute(schema_sql)

#     # Infer table order if not provided
#     if table_order is None:
#         table_order = infer_table_order(schema_sql)
#         print(f"Table order inferred: {table_order}")
    
#     # Import data from Parquet files in the inferred or provided order
#     for table in table_order:
#         if table in table_names:
#             parquet_file = export_dir / f"{table}.parquet"
#             new_conn.execute(f"INSERT INTO {table} SELECT * FROM parquet_scan('{parquet_file}')")

            

#     # Import any remaining tables that weren't in the specified order
#     for table in table_names:
#         if table not in table_order:
#             parquet_file = export_dir / f"{table}.parquet"
#             new_conn.execute(f"INSERT INTO {table} SELECT * FROM parquet_scan('{parquet_file}')")

#     new_conn.close()

#     print(f"Database exported and imported successfully. New database: {new_database}")


#     return new_database

# # This method doesn't work as MotherDuck bug


In [5]:
def copy_database_to_motherduck(source_db_path, new_db_name, tables=None):
    conn = duckdb.connect(':memory:')
    load_dotenv()

    md_token = os.getenv('MD_TOKEN')
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")
        
    conn.sql("ATTACH 'md:';")

    try:
        # Attach the source database
        print(f"ATTACH '{source_db_path}' AS local_source_db")
        conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

        print("USE 'local_source_db'")
        conn.execute("USE 'local_source_db'")

            # Get the list of tables from the source database
        if tables is None:
            tables = [table[0] for table in conn.execute("SHOW TABLES").fetchall()]
        
        # Create the new database
        print(f"CREATE OR REPLACE DATABASE {new_db_name}")
        conn.execute(f"CREATE OR REPLACE DATABASE {new_db_name}")
        

        # Copy the schema from the source database to the new database
        print(f"COPY FROM DATABASE local_source_db TO {new_db_name} (SCHEMA)")
        conn.execute(f"COPY FROM DATABASE local_source_db TO {new_db_name} (SCHEMA)")
        

        # Copy data for each table
        print(tables)
        for table_name in tables:
            print(f"INSERT INTO {new_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}")
            conn.execute(f"INSERT INTO {new_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}")
        
        print(f"Database '{new_db_name}' created and populated successfully.")
    
    except Exception as e:
        print(f"An error occurred: {str(e)}")
    
    finally:
        conn.close()

    return new_db_name

# Usage example
# source_db_path = 'data/Experiments_CB_2023_Q4.ddb'
# new_db_name = 'cb_21Oct2024'
# copy_database(source_db_path, new_db_name)

In [6]:
# Example usage:
# export_import_duckdb('original.db', 'new.db', export_dir='my_export', location='local', schema_file='schema.sql')
# Or with custom table order:
# export_import_duckdb('original.db', 'new.db', export_dir='my_export', location='local', schema_file='schema.sql', table_order=['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs'])


In [7]:
CB_DATABASE = Path("../data/Experiments_CB_2023_Q4.ddb")

In [8]:
CB_DATABASE_MD = str(CB_DATABASE.stem)

In [9]:
# EXPORT_DIR = Path("../data/parquet")

In [10]:
# DB_SCHEMA = Path("../sql/create_cb_experiment_tables_ddb.sql")

In [11]:
new_db = copy_database_to_motherduck(CB_DATABASE, CB_DATABASE_MD, table_order)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ATTACH '../data/Experiments_CB_2023_Q4.ddb' AS local_source_db
USE 'local_source_db'
CREATE OR REPLACE DATABASE Experiments_CB_2023_Q4
COPY FROM DATABASE local_source_db TO Experiments_CB_2023_Q4 (SCHEMA)
['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']
INSERT INTO Experiments_CB_2023_Q4.ExperimentCBs SELECT * FROM local_source_db.ExperimentCBs
INSERT INTO Experiments_CB_2023_Q4.TrialCBs SELECT * FROM local_source_db.TrialCBs
INSERT INTO Experiments_CB_2023_Q4.ResponseCBs SELECT * FROM local_source_db.ResponseCBs
INSERT INTO Experiments_CB_2023_Q4.EventCBs SELECT * FROM local_source_db.EventCBs
Database 'Experiments_CB_2023_Q4' created and populated successfully.


In [12]:
# new_db = export_import_duckdb(
#     CB_DATABASE,
#     Path(str(CB_DATABASE).replace(".ddb", "_FROM_PARQUET.ddb")),
#     export_dir=EXPORT_DIR,
#     location="local",
#     schema_file=DB_SCHEMA,
#     table_order=table_order,
# )

In [13]:
def compare_databases(original_db, new_db):
    conn_original = duckdb.connect(original_db)
    conn_new = duckdb.connect(new_db)

    tables_original = conn_original.execute("SHOW TABLES").fetchall()
    tables_new = conn_new.execute("SHOW TABLES").fetchall()

    if len(tables_original) != len(tables_new):
        print(f"Number of tables mismatch. Original: {len(tables_original)}, New: {len(tables_new)}")
        return False

    for table in tables_original:
        table_name = table[0]
        print(f"Comparing table: {table_name}")

        schema_original = conn_original.execute(f"DESCRIBE {table_name}").fetchall()
        schema_new = conn_new.execute(f"DESCRIBE {table_name}").fetchall()

        if schema_original != schema_new:
            print(f"Schema mismatch for table {table_name}")
            print("Original schema:")
            print(pd.DataFrame(schema_original, columns=['Column', 'Type', 'Null', 'Key', 'Default', 'Extra']))
            print("\nNew schema:")
            print(pd.DataFrame(schema_new, columns=['Column', 'Type', 'Null', 'Key', 'Default', 'Extra']))
            return False

        # Compare data
        data_original = conn_original.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()
        data_new = conn_new.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()

        if data_original != data_new:
            print(f"Data mismatch for table {table_name}")
            
            # For TrialCBs table, perform a more detailed comparison
            if table_name == 'TrialCBs':
                df_original = pd.DataFrame(data_original, columns=[col[0] for col in schema_original])
                df_new = pd.DataFrame(data_new, columns=[col[0] for col in schema_new])
                
                # Compare row counts
                if len(df_original) != len(df_new):
                    print(f"Row count mismatch. Original: {len(df_original)}, New: {len(df_new)}")
                
                # Compare column values
                for column in df_original.columns:
                    if not df_original[column].equals(df_new[column]):
                        print(f"Mismatch in column: {column}")
                        print("First few mismatching rows:")
                        mismatch = (df_original[column] != df_new[column])
                        print(pd.concat([df_original[mismatch], df_new[mismatch]], axis=1, keys=['Original', 'New']).head())
                
            return False

        # Compare row count
        count_original = conn_original.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        count_new = conn_new.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

        if count_original != count_new:
            print(f"Row count mismatch for table {table_name}. Original: {count_original}, New: {count_new}")
            return False

    print("All tables and data match!")
    return True

In [14]:
# Usage
# original_db = "path/to/original.db"
# new_db = "path/to/new.db"
# are_identical = compare_databases(original_db, new_db)
# print(f"Databases are identical: {are_identical}")

In [15]:
are_identical = compare_databases(CB_DATABASE, new_db)
print(f"Databases are identical: {are_identical}")

Number of tables mismatch. Original: 4, New: 0
Databases are identical: False
